In [1]:
# weather + accidents

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
# spark = SparkSession.builder.appName("NYC").config("spark.jars", "/jars/postgresql-42.7.3.jar").getOrCreate()
# spark = SparkSession.builder.appName("NYC").config("spark.jars.packages", "org.postgresql:postgresql:42.7.3").getOrCreate()
spark = SparkSession.builder.appName("NYC_Bad_Weather").getOrCreate()

base_path = "/home/jovyan/work"

In [4]:
# spark._jvm.java.lang.Class.forName("org.postgresql.Driver")

In [5]:
weather_df = spark.read.parquet(f"{base_path}/CLEANED_weather_partitioned/")
collision_df = spark.read.parquet(f"{base_path}/CLEANED_vehicle_collisions_partitioned/")

weather_df_filtered = weather_df.filter((weather_df.year == 2023) | (weather_df.year == 2024) | (weather_df.year == 2025))
collision_df_filtered = collision_df.filter((collision_df.year == 2023) | (collision_df.year == 2024) | (collision_df.year == 2025))

In [6]:
#weather_df_filtered.sort("Date", ascending=True) #.show(3, vertical=True)

In [7]:
#collision_df_filtered.sort("CRASH DATE", ascending=True) #.show(3, vertical=True)

In [8]:
weather_hourly = weather_df_filtered.withColumn(
    "hourly_precip_num",
    when(col("HourlyPrecipitation") == "T", 0.001)
    .otherwise(col("HourlyPrecipitation").cast("double"))
)

weather_hourly = weather_hourly.withColumn(
    "weather_condition",
    when(col("hourly_precip_num") > 0, "Rain")
    .otherwise("No Rain")
)

In [9]:
# Cuts "11:00" at ":" -> Keeps 11
collision_df_filtered = collision_df_filtered.withColumn(
    "crash_hour", 
    split(col("CRASH TIME"), ":")[0].cast("int")
)

# Cuts "23:51:00" at ":"-> Keeps 23
weather_hourly = weather_hourly.withColumn(
    "weather_hour", 
    split(col("Time"), ":")[0].cast("int")
)

joined_df = collision_df_filtered.join(
    weather_hourly.select("Date", "weather_hour", "weather_condition"),
    (collision_df_filtered["CRASH DATE"] == weather_hourly["Date"]) & (collision_df_filtered["crash_hour"] == weather_hourly["weather_hour"]),
    "inner"
)

In [10]:
weather_daily = weather_hourly.groupBy("Date").agg(
    max(
        when(col("hourly_precip_num") > 0, 1)
        .otherwise(0)
    ).alias("rain_day")
)

daily_collisions = collision_df_filtered.groupBy(
    col("CRASH DATE").alias("Date")
).agg(
    count("*").alias("daily_collisions")
)

daily_joined = daily_collisions.join(
    weather_daily,
    "Date",
    "inner"
)

daily_joined = daily_joined.withColumn("year", year("Date")).withColumn("month", month("Date"))

In [11]:
# daily_result = daily_joined.groupBy("rain_day").agg(
#     avg("daily_collisions").alias("avg_collisions_per_day"),
#     count("*").alias("num_days")
# )

daily_result = daily_joined.groupBy("year", "month", "rain_day").agg(avg("daily_collisions").alias("avg_collisions_per_day"), count("*").alias("num_days")).orderBy("year", "month", "rain_day")

daily_result.show()

+----+-----+--------+----------------------+--------+
|year|month|rain_day|avg_collisions_per_day|num_days|
+----+-----+--------+----------------------+--------+
|2023|    1|       0|     206.9090909090909|      11|
|2023|    1|       1|                234.65|      20|
|2023|    2|       0|              234.3125|      16|
|2023|    2|       1|                 222.0|      12|
|2023|    3|       0|    238.66666666666666|      15|
|2023|    3|       1|               247.625|      16|
|2023|    4|       0|                 230.7|      20|
|2023|    4|       1|                 252.5|      10|
|2023|    5|       0|                 268.5|      22|
|2023|    5|       1|    250.55555555555554|       9|
|2023|    6|       0|    249.57142857142858|      14|
|2023|    6|       1|              256.5625|      16|
|2023|    7|       0|              246.1875|      16|
|2023|    7|       1|                 245.0|      15|
|2023|    8|       0|     237.6153846153846|      13|
|2023|    8|       1|    240

In [12]:
hour_counts = weather_hourly.groupBy("weather_condition", "weather_hour").agg(count("*").alias("total_hours"))

collision_counts = joined_df.groupBy("weather_condition", "weather_hour").agg(count("*").alias("total_collisions"))


In [13]:
#collisions in rainy vs dry hours
hourly_result = collision_counts.join(hour_counts, ["weather_condition", "weather_hour"]).withColumn("collisions_per_hour", col("total_collisions") / col("total_hours")).orderBy("weather_hour", "weather_condition")

hourly_result = collision_counts.join(hour_counts, ["weather_condition", "weather_hour"]).withColumn("collisions_per_hour", col("total_collisions") / col("total_hours")).orderBy("weather_hour", "weather_condition")

hourly_result.show(48)

+-----------------+------------+----------------+-----------+-------------------+
|weather_condition|weather_hour|total_collisions|total_hours|collisions_per_hour|
+-----------------+------------+----------------+-----------+-------------------+
|          No Rain|           0|           11937|       1125| 10.610666666666667|
|             Rain|           0|            3620|        310|  11.67741935483871|
|          No Rain|           1|            6249|       1115|  5.604484304932735|
|             Rain|           1|            2054|        293|  7.010238907849829|
|          No Rain|           2|            5223|       1158|  4.510362694300518|
|             Rain|           2|            1358|        279|  4.867383512544803|
|          No Rain|           3|            4706|       1148|  4.099303135888502|
|             Rain|           3|            1465|        295|  4.966101694915254|
|          No Rain|           4|            5189|       1120|  4.633035714285715|
|             Ra

In [14]:
daily_result.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "daily_AVG_weather_accidents") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [15]:
hourly_result.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "hourly_weather_accidents") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [16]:
# df_check = spark.read \
#     .format("jdbc") \
#     .option("url", "jdbc:postgresql://postgres:5432/traffic") \
#     .option("dbtable", "hourly_weather_accidents") \
#     .option("user", "admin") \
#     .option("password", "admin") \
#     .option("driver", "org.postgresql.Driver") \
#     .load()

# df_check.show()